In [1]:
!apt-get update -qq
!apt-get install -y ns2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libotcl1 libpcap0.8t64 libtclcl1
Suggested packages:
  gnuplot
The following NEW packages will be installed:
  libotcl1 libpcap0.8t64 libtclcl1 ns2
0 upgraded, 4 newly installed, 0 to remove and 68 not upgraded.
Need to get 2,810 kB of archives.
After this operation, 15.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libpcap0.8t64 amd64 1.10.4-4.1ubuntu3.1 [151 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 libotcl1 amd64 1.14+dfsg-8 [24.2 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 libtclcl1 amd64 1.20-10 [65.2 kB]
Get:4 http://archive.ubuntu.com/ubuntu n

In [2]:
%%writefile exp4.tcl

set ns [new Simulator]

set tr [open out.tr w]
set nf [open out.nam w]

$ns trace-all $tr
$ns namtrace-all $nf

# Create four nodes
set n0 [$ns node]
set n1 [$ns node]
set n2 [$ns node]
set n3 [$ns node]

# Create Ethernet LAN
$ns make-lan "$n0 $n1 $n2 $n3" 10Mb 10ms LL Queue/DropTail Mac/802_3

# TCP connection
set tcp [$ns create-connection TCP $n0 TCPSink $n3 0]

# FTP application
set ftp [new Application/FTP]
$ftp attach-agent $tcp

# Start and stop FTP
$ns at 0.5 "$ftp start"
$ns at 4.5 "$ftp stop"

# Finish simulation
$ns at 5.0 "finish"

proc finish {} {
    global ns tr nf
    $ns flush-trace
    close $tr
    close $nf
    exit 0
}

$ns run

Writing exp4.tcl


In [3]:
!ns exp4.tcl


	see tcl-object.tcl in tclcl for info about this warning.



In [4]:
!ls

exp4.tcl  out.nam  out.tr  sample_data


In [5]:
!head out.tr

h 0.5 0 4 tcp 40 ------- 0 0.0 3.0 0 0
+ 0.51 0 4 tcp 40 ------- 0 0.0 3.0 0 0
- 0.51 0 4 tcp 40 ------- 0 0.0 3.0 0 0
r 0.520055 4 3 tcp 40 ------- 0 0.0 3.0 0 0
h 0.520055 3 4 ack 40 ------- 0 3.0 0.0 0 1
+ 0.530055 3 4 ack 40 ------- 0 3.0 0.0 0 1
- 0.530055 3 4 ack 40 ------- 0 3.0 0.0 0 1
r 0.54011 4 0 ack 40 ------- 0 3.0 0.0 0 1
h 0.54011 0 4 tcp 1040 ------- 0 0.0 3.0 1 2
h 0.54011 0 4 tcp 1040 ------- 0 0.0 3.0 2 3


In [6]:
!awk '$1=="r" {n++} END {print "Received packets:",n+0}' out.tr

Received packets: 3782


In [7]:
!awk '$1=="d" {n++} END {print "Dropped packets:",n+0}' out.tr

Dropped packets: 0
